# Carbon Ledger Quantum v8
Upload the complete project ZIP as a Kaggle Dataset, attach it, enable Internet, and run all cells. Small experiments use CPU. Results are actual synthetic experiments; no guaranteed quantum advantage. The original website model is preserved unless a candidate passes the gate.

In [ ]:
from pathlib import Path
import zipfile, os
archives = [p for p in Path('/kaggle/input').rglob('*.zip') if 'carbon' in p.name.lower()]
if len(archives) != 1:
    raise RuntimeError(f'Attach exactly one Carbon Ledger project ZIP, found {len(archives)}')
dest = Path('/kaggle/working/carbon-ledger-v8'); dest.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as z:
    for member in z.infolist():
        if not (dest / member.filename).resolve().is_relative_to(dest.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(dest)
roots = [p.parent for p in dest.rglob('package.json') if (p.parent / 'quantum_lab').exists()]
if len(roots) != 1: raise RuntimeError('Project root not found')
os.chdir(roots[0]); print('Project:', Path.cwd())


In [ ]:
%pip install -q -r quantum_lab/requirements.txt


In [ ]:
import subprocess, sys
# Increase these only after the small run completes.
subprocess.run([sys.executable, '-m', 'quantum_lab.benchmark', '--scenarios', '8', '--budget', '100'], check=True)
subprocess.run([sys.executable, '-m', 'quantum_lab.risk', '--samples', '300', '--iterations', '120'], check=True)


In [ ]:
# Independent validation/test split; only a passing candidate is promoted.
subprocess.run([sys.executable, '-m', 'quantum_lab.surrogate', '--samples', '1200', '--promote'], check=True)


In [ ]:
import json, pandas as pd
risk = json.loads(Path('public/data/quantum-risk-v8.json').read_text())
display(pd.DataFrame(risk['rows']).drop(columns=['confusion','curves']))
display(json.loads(Path('artifacts/quantum-lab/surrogate-validation.json').read_text()))


In [ ]:
output = Path('/kaggle/working/Carbon-Ledger-v8-Training-Results.zip')
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ['public/data', 'public/models', 'artifacts/quantum-lab']:
        for p in Path(folder).rglob('*'):
            if p.is_file(): z.write(p, str(p))
print('Download:', output)
